# Notebook 3:
### Hourly Workload and Staffing Coverage Model

**Purpose**

Transforms prescription activity into hourly workload, estimates required staffing, expands weekly schedules into hourly coverage, and computes staffing gaps.

#### Output
- `projected_staffing_by_hour.csv`

**Key responsibilitiy**
- Provides the operational efficiency layer used to evaluate staffing alignment against prescription workload in Power BI.

In [1]:
# --- Config (paths) ---
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "bi_outputs"
FORECAST_DIR = PROJECT_ROOT / "forecast"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUT_DIR       =", OUT_DIR)
print("FORECAST_DIR  =", FORECAST_DIR)


PROJECT_ROOT = /Users/selenadavis/PythonProject/Notebooks
OUT_DIR       = /Users/selenadavis/PythonProject/Notebooks/bi_outputs
FORECAST_DIR  = /Users/selenadavis/PythonProject/Notebooks/forecast


In [2]:
# --- 0) Imports + parameters ---
from pathlib import Path
import numpy as np
import pandas as pd

# Operating hours (inclusive)
OPEN_HOUR = 8
CLOSE_HOUR = 17

# Staffing model knobs
MIN_STAFF = 1.0

# Throughput assumption: Rx per staff-hour.
# Higher value -> LOWER required staff. Calibrate to real ops.
RX_PER_STAFF_PER_HOUR = 4.0

# Smoothing window (hours) for display only
SMOOTH_WINDOW = 3

SEED = 42
rng = np.random.default_rng(SEED)

OUT_DIR = Path("bi_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "projected_staffing_by_hour.csv"
print("OUT_FILE =", OUT_FILE.resolve())

OUT_FILE = /Users/selenadavis/PythonProject/Notebooks/bi_outputs/projected_staffing_by_hour.csv


In [3]:
# --- 1) Locate input files (prescriptions + staffschedule) ---
def find_first_existing(candidates):
    for p in candidates:
        pp = Path(p)
        if pp.exists():
            return pp
    return None

PRESCRIPTIONS_CSV = find_first_existing([
    "prescriptions.csv",
    "data/prescriptions.csv",
    "../prescriptions.csv",
    "../data/prescriptions.csv",
    str(OUT_DIR / "prescriptions.csv"),
])

STAFF_CSV = find_first_existing([
    "staffschedule.csv",
    "data/staffschedule.csv",
    "../staffschedule.csv",
    "../data/staffschedule.csv",
    str(OUT_DIR / "staffschedule.csv"),
])

print("Inputs found:")
print(" - prescriptions:", PRESCRIPTIONS_CSV.name if PRESCRIPTIONS_CSV else "MISSING")
print(" - staffschedule:", STAFF_CSV.name if STAFF_CSV else "MISSING")

if PRESCRIPTIONS_CSV is None:
    raise FileNotFoundError("Could not find prescriptions.csv in common locations.")
if STAFF_CSV is None:
    raise FileNotFoundError("Could not find staffschedule.csv in common locations.")

print("Paths:")
print(" -", PRESCRIPTIONS_CSV.resolve())
print(" -", STAFF_CSV.resolve())

Inputs found:
 - prescriptions: prescriptions.csv
 - staffschedule: staffschedule.csv
Paths:
 - /Users/selenadavis/PythonProject/Notebooks/prescriptions.csv
 - /Users/selenadavis/PythonProject/Notebooks/staffschedule.csv


In [4]:
# --- 2) Load prescriptions and build hourly workload (RxCount) ---
def detect_first(df: pd.DataFrame, candidates: list[str], what: str) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No {what} column found. Tried: {candidates}. Got columns: {df.columns.tolist()}")

rx = pd.read_csv(PRESCRIPTIONS_CSV, low_memory=False)

date_col = next((c for c in ["FillDate", "RefillDate", "Date", "Timestamp"] if c in rx.columns), None)
if date_col is None:
    raise ValueError("prescriptions.csv must include FillDate or RefillDate or Date")

_ = detect_first(rx, ["QuantityDispensed", "Quantity", "Qty", "Units"], "quantity") if any(c in rx.columns for c in ["QuantityDispensed","Quantity","Qty","Units"]) else None

ts = pd.to_datetime(rx[date_col], errors="coerce")
rx = rx.loc[ts.notna()].copy()
rx["Timestamp"] = ts.loc[ts.notna()].values

rx["Date"] = pd.to_datetime(rx["Timestamp"]).dt.normalize()
hours = pd.to_datetime(rx["Timestamp"]).dt.hour

# If all times are midnight, synthesize a plausible operating-hour distribution
if hours.nunique() == 1 and int(hours.iloc[0]) == 0:
    hour_bins = np.arange(OPEN_HOUR, CLOSE_HOUR + 1)
    # heavier mid-day, lighter open/close
    center = (OPEN_HOUR + CLOSE_HOUR) / 2
    sigma = 2.2
    raw = np.exp(-0.5 * ((hour_bins - center) / sigma) ** 2)
    raw = raw / raw.sum()
    rx["Hour"] = rng.choice(hour_bins, size=len(rx), replace=True, p=raw)
else:
    rx["Hour"] = hours.astype(int)

# Restrict to operating hours
rx = rx[(rx["Hour"] >= OPEN_HOUR) & (rx["Hour"] <= CLOSE_HOUR)].copy()

# Hourly workload (count of prescriptions)
hourly_req = (
    rx.groupby(["Date","Hour"], as_index=False)
      .size()
      .rename(columns={"size":"RxCount"})
)

# Complete Date x Hour grid for continuous visuals
all_days = pd.date_range(hourly_req["Date"].min(), hourly_req["Date"].max(), freq="D")
all_hours = pd.DataFrame({"Hour": list(range(OPEN_HOUR, CLOSE_HOUR + 1))})

grid = (
    pd.DataFrame({"Date": all_days}).assign(key=1)
    .merge(all_hours.assign(key=1), on="key")
    .drop(columns="key")
)
grid["Date"] = pd.to_datetime(grid["Date"]).dt.normalize()
grid["Hour"] = grid["Hour"].astype(int)

hourly_req = grid.merge(hourly_req, on=["Date","Hour"], how="left")
hourly_req["RxCount"] = hourly_req["RxCount"].fillna(0).astype(int)

# Smoothed demand for trending only
hourly_req = hourly_req.sort_values(["Date","Hour"]).copy()
hourly_req["RxCount_Smoothed"] = (
    hourly_req.groupby("Date")["RxCount"]
              .transform(lambda s: s.rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean())
)

# Required staffing from RAW demand
hourly_req["RequiredStaff"] = np.maximum(MIN_STAFF, hourly_req["RxCount"] / RX_PER_STAFF_PER_HOUR)

print("Rx date range:", hourly_req["Date"].min().date(), "to", hourly_req["Date"].max().date())
print("Avg required:", float(hourly_req["RequiredStaff"].mean()))
hourly_req.head()

Rx date range: 2023-01-01 to 2024-12-31
Avg required: 4.14969220246238


,Date,Hour,RxCount,RxCount_Smoothed,RequiredStaff
0,2023-01-01,8,5,8.000000,1.25
1,2023-01-01,9,11,11.000000,2.75
2,2023-01-01,10,17,17.333333,4.25
3,2023-01-01,11,24,22.333333,6.00
4,2023-01-01,12,26,28.666667,6.50


In [5]:
# --- 3) Load staffschedule and build scheduled staffing by hour ---
staff = pd.read_csv(STAFF_CSV, low_memory=False)

# Detect columns
week_col = next((c for c in ["WeekStartDate","WeekStart","WeekStart_Date","Week"] if c in staff.columns), None)
hours_col = next((c for c in ["HoursWorked","Hours","TotalHours","WeeklyHours"] if c in staff.columns), None)
shift_col = next((c for c in ["Shift","ShiftBucket","ShiftType"] if c in staff.columns), None)

if week_col is None or hours_col is None:
    raise ValueError(f"staffschedule.csv must include a week column and hours column. Found columns: {staff.columns.tolist()}")

staff = staff.copy()
staff[week_col] = pd.to_datetime(staff[week_col], errors="coerce").dt.normalize()
staff[hours_col] = pd.to_numeric(staff[hours_col], errors="coerce")

staff = staff.loc[staff[week_col].notna()].copy()
staff = staff.loc[staff[hours_col].notna()].copy()

# Normalize shift labels if present
if shift_col is not None:
    staff[shift_col] = staff[shift_col].astype(str).str.upper().str.strip()

# Weekly totals
weekly_total = (
    staff.groupby(week_col, as_index=False)
         .agg(WeeklyHoursTotal=(hours_col, "sum"))
)

# Weekly totals by shift bucket
if shift_col is not None:
    weekly_by_shift = (
        staff.groupby([week_col, shift_col], as_index=False)
             .agg(WeeklyHoursShift=(hours_col, "sum"))
    )
else:
    weekly_by_shift = None

# Build schedule grid matching the Rx date range
dates = pd.date_range(hourly_req["Date"].min(), hourly_req["Date"].max(), freq="D")
hours = list(range(OPEN_HOUR, CLOSE_HOUR + 1))

sched_grid = (
    pd.DataFrame({"Date": dates}).assign(key=1)
    .merge(pd.DataFrame({"Hour": hours}).assign(key=1), on="key")
    .drop(columns="key")
)
sched_grid["Date"] = pd.to_datetime(sched_grid["Date"]).dt.normalize()
sched_grid["Hour"] = sched_grid["Hour"].astype(int)

# Map each Date to its WeekStartDate (Monday)
sched_grid["WeekStartDate"] = (sched_grid["Date"] - pd.to_timedelta(sched_grid["Date"].dt.weekday, unit="D")).dt.normalize()

# Operating-hours shape: heavier mid-day
hour_bins = np.arange(OPEN_HOUR, CLOSE_HOUR + 1)
center = (OPEN_HOUR + CLOSE_HOUR) / 2
sigma = 2.2
raw = np.exp(-0.5 * ((hour_bins - center) / sigma) ** 2)
raw = raw / raw.sum()
hour_weight_map = dict(zip(hour_bins, raw))
sched_grid["HourWeight"] = sched_grid["Hour"].map(hour_weight_map).astype(float)

# Join weekly totals
sched_grid = sched_grid.merge(weekly_total.rename(columns={week_col:"WeekStartDate"}), on="WeekStartDate", how="left")

# Fallback for missing weeks: median weekly hours
default_weekly_total = float(weekly_total["WeeklyHoursTotal"].median()) if len(weekly_total) else 0.0
mask_missing_week = sched_grid["WeeklyHoursTotal"].isna() | (sched_grid["WeeklyHoursTotal"] <= 0)

sched_grid["ScheduleSource"] = np.where(mask_missing_week, "Fallback", "StaffSchedule")
sched_grid.loc[mask_missing_week, "WeeklyHoursTotal"] = default_weekly_total

# Daily staff-hours = weekly / 7
sched_grid["DailyHoursTotal"] = sched_grid["WeeklyHoursTotal"] / 7.0

# If shift detail exists, allocate by shift proportions; otherwise uniform (1.0)
sched_grid["ShiftProp"] = 1.0

sched_grid["HourShare"] = sched_grid.groupby("Date")["HourWeight"].transform(lambda s: s / s.sum())

# Scheduled staff per hour (staff-hours allocated to a 1-hour bucket)
sched_grid["ScheduledStaff"] = sched_grid["DailyHoursTotal"] * sched_grid["ShiftProp"] * sched_grid["HourShare"]

# Build merge-ready schedule table (1 row per Date-Hour)
sched = (
    sched_grid[["Date", "Hour", "ScheduledStaff", "ScheduleSource"]]
    .groupby(["Date", "Hour"], as_index=False)
    .agg(
        ScheduledStaff=("ScheduledStaff", "sum"),
        ScheduleSource=("ScheduleSource", "first")
    )
)

print("Avg scheduled:", float(sched["ScheduledStaff"].mean()))
print("ScheduleSource counts:")
print(sched["ScheduleSource"].value_counts(dropna=False))
sched.head()

Avg scheduled: 3.6406931795974207
ScheduleSource counts:
ScheduleSource
StaffSchedule    7310
Name: count, dtype: int64


,Date,Hour,ScheduledStaff,ScheduleSource
0,2023-01-01,8,0.457759,StaffSchedule
1,2023-01-01,9,1.046063,StaffSchedule
2,2023-01-01,10,1.944234,StaffSchedule
3,2023-01-01,11,2.939064,StaffSchedule
4,2023-01-01,12,3.613594,StaffSchedule


In [6]:
# --- 4) Merge required vs scheduled, compute deltas, export ---
merged = hourly_req.merge(sched, on=["Date","Hour"], how="left")

merged["ScheduledStaff"] = merged["ScheduledStaff"].fillna(0.0)
merged["ScheduleSource"] = merged["ScheduleSource"].fillna("Missing")

# Headcount proxy for display: ceil of staff per hour
merged["ScheduledHeadcount"] = np.ceil(merged["ScheduledStaff"]).astype(float)

merged["OverUnder"] = merged["ScheduledStaff"] - merged["RequiredStaff"]
merged["OverUnderFlag"] = np.where(merged["OverUnder"] < 0, "Understaffed", "Overstaffed")

under_hours = int((merged["OverUnder"] < 0).sum())
print("Avg required:", float(merged["RequiredStaff"].mean()))
print("Avg scheduled:", float(merged["ScheduledStaff"].mean()))
print("Understaffed hours:", under_hours, "out of", len(merged))

# Order columns for Power BI
out_cols = [
    "Date","Hour",
    "RxCount","RxCount_Smoothed",
    "RequiredStaff",
    "ScheduledStaff","ScheduledHeadcount",
    "ScheduleSource",
    "OverUnder","OverUnderFlag"
]

merged[out_cols].to_csv(OUT_FILE, index=False)
print("Wrote:", OUT_FILE.resolve(), merged[out_cols].shape)

merged[out_cols].head(12)

Avg required: 4.14969220246238
Avg scheduled: 3.6406931795974207
Understaffed hours: 5382 out of 7310
Wrote: /Users/selenadavis/PythonProject/Notebooks/bi_outputs/projected_staffing_by_hour.csv (7310, 10)


,Date,Hour,RxCount,RxCount_Smoothed,RequiredStaff,ScheduledStaff,ScheduledHeadcount,ScheduleSource,OverUnder,OverUnderFlag
0,2023-01-01,8,5,8.000000,1.25,0.457759,1.0,StaffSchedule,-0.792241,Understaffed
1,2023-01-01,9,11,11.000000,2.75,1.046063,2.0,StaffSchedule,-1.703937,Understaffed
2,2023-01-01,10,17,17.333333,4.25,1.944234,2.0,StaffSchedule,-2.305766,Understaffed
3,2023-01-01,11,24,22.333333,6.00,2.939064,3.0,StaffSchedule,-3.060936,Understaffed
4,2023-01-01,12,26,28.666667,6.50,3.613594,4.0,StaffSchedule,-2.886406,Understaffed
5,2023-01-01,13,36,31.333333,9.00,3.613594,4.0,StaffSchedule,-5.386406,Understaffed
6,2023-01-01,14,32,26.000000,8.00,2.939064,3.0,StaffSchedule,-5.060936,Understaffed
7,2023-01-01,15,10,16.666667,2.50,1.944234,2.0,StaffSchedule,-0.555766,Understaffed
8,2023-01-01,16,8,6.333333,2.00,1.046063,2.0,StaffSchedule,-0.953937,Understaffed
9,2023-01-01,17,1,4.500000,1.00,0.457759,1.0,StaffSchedule,-0.542241,Understaffed
